# 🧠 Hallucination Detector — GRPO Fine-Tuning with Qwen3-0.6B

## Group Relative Policy Optimization for Hallucination Detection

This notebook fine-tunes **Qwen3-0.6B** using **GRPO** (Group Relative Policy Optimization) 
from the TRL library to detect, classify, and correct hallucinations in LLM-generated text.

### Why GRPO?
GRPO replaces the value function with group-relative advantage estimation, removing the critic and enabling more stable, compute-efficient large-scale LLM fine-tuning.

### Pipeline Overview
| Step | What | Why |
|------|------|-----|
| 1 | Build reward-labeled dataset from gym tasks | Ground-truth for GRPO rewards |
| 2 | Define reward functions (gym RewardEngine) | Dense, multi-signal rewards |
| 3 | Baseline evaluation | Measure pre-training performance |
| 4 | GRPO training loop | Fine-tune the model weights |
| 5 | Post-training evaluation & comparison | Measure improvement |
| 6 | Plot publication-quality graphs | Visualize training progress |
| 7 | Push trained model to Hugging Face | Share the results |

### Model Choice: `Qwen/Qwen3-0.6B`
- **Qwen3** — latest generation, with built-in thinking/non-thinking modes and strong agent capabilities

In [1]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1 — Imports & Environment Setup
# ══════════════════════════════════════════════════════════════════════════════
import os, sys, json, re, time, warnings, textwrap
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm.auto import tqdm

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import GRPOTrainer, GRPOConfig
import peft, trl

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="husl", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 150, "font.family": "sans-serif"})

# ── Auto-detect project root (works from any CWD) ────────────────────────────
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "hallucination_detector_gym").exists():
    # Fallback: assume notebook is in the project root
    PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# ── Load .env for HF_TOKEN / HF_USERNAME ─────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# ── Configuration — edit these for your own setup ─────────────────────────────
MODEL_ID      = "Qwen/Qwen3-0.6B"                  # Base model (ungated, Apache 2.0)
HF_USERNAME   = os.getenv("HF_USERNAME", "")        # Your HF username (from .env or set here)
HF_REPO_NAME  = "hallucination-detector-agent-qwen3-0.6b"
MODEL_REPO    = f"{HF_USERNAME}/{HF_REPO_NAME}" if HF_USERNAME else ""

# ── Auto-detect best available device ─────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"✅ PyTorch {torch.__version__}  |  Device: {DEVICE}")
print(f"   TRL: {trl.__version__}  |  PEFT: {peft.__version__}")
print(f"📁 Project: {PROJECT_ROOT}")
print(f"🤗 HF repo: {MODEL_REPO or '(not configured — set HF_USERNAME in .env)'}")

✅ PyTorch 2.11.0  |  Device: mps
   TRL: 1.0.0  |  PEFT: 0.18.1
📁 Project: /Users/williyam-23540/Desktop/hallucination-detector-gym
🤗 HF repo: williyam/hallucination-detector-agent-qwen3-0.6b


## 📋 Step 1 — Build GRPO Training Dataset

GRPO needs a dataset of **prompts**. For each prompt the model generates multiple completions; 
our reward function scores each one using the gym's `RewardEngine`. We construct diverse 
prompts from all 3 tasks with varied difficulty and instruction styles.

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2 — Build GRPO training dataset from gym tasks
# ══════════════════════════════════════════════════════════════════════════════
from hallucination_detector_gym.tasks import list_tasks, get_task, TASK_REGISTRY
from hallucination_detector_gym.constants import TaskID, HallucinationType

tasks = list_tasks()

# ── System instruction for the hallucination detector ────────────────────────
SYSTEM_MSG = (
    "You are a hallucination detector. Given a SOURCE (ground truth) and a PASSAGE "
    "(LLM-generated), output a JSON object with:\n"
    '- "action_type": one of "detect", "classify", "correct", "submit"\n'
    '- "hallucination_detected": true/false\n'
    '- "hallucination_type": "factual_error" | "entity_fabrication" | "logical_inconsistency" | "none"\n'
    '- "hallucinated_span": exact substring from the passage\n'
    '- "corrected_text": the correct replacement\n'
    '- "reasoning": your analysis\n'
    "Respond with ONLY valid JSON."
)

# ── Prompt templates (varied for robustness) ─────────────────────────────────
TEMPLATES = [
    # Template A: detect
    "Detect hallucinations.\n\nSOURCE:\n{source}\n\nPASSAGE:\n{passage}\n\n"
    "Respond with a JSON action: detect any hallucination, include the exact hallucinated_span.",
    # Template B: classify
    "Classify the hallucination type.\n\nSOURCE:\n{source}\n\nPASSAGE:\n{passage}\n\n"
    "Respond with a JSON action: classify the hallucination type for the span \"{span}\".",
    # Template C: correct
    "Correct the hallucination.\n\nSOURCE:\n{source}\n\nPASSAGE:\n{passage}\n\n"
    "The span \"{span}\" is a {htype}. Respond with a JSON action to correct it.",
    # Template D: full pipeline
    "Analyse this passage for ALL hallucinations.\n\nSOURCE:\n{source}\n\nPASSAGE:\n{passage}\n\n"
    "For each hallucination: detect it, classify it, and correct it. Respond with a JSON action for the first one.",
]

# ── Build prompts ────────────────────────────────────────────────────────────
rows = []
for task in tasks:
    src = task.source_context
    psg = task.passage
    for ann in task.annotations:
        # Detect prompt
        rows.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_MSG},
                {"role": "user", "content": TEMPLATES[0].format(source=src, passage=psg)},
            ],
            "task_id": task.task_id.value,
            "expected_action": "detect",
            "expected_span": ann.hallucinated_span,
            "expected_type": ann.hallucination_type.value,
            "expected_correction": ann.corrected_text,
        })
        # Classify prompt
        rows.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_MSG},
                {"role": "user", "content": TEMPLATES[1].format(
                    source=src, passage=psg, span=ann.hallucinated_span)},
            ],
            "task_id": task.task_id.value,
            "expected_action": "classify",
            "expected_span": ann.hallucinated_span,
            "expected_type": ann.hallucination_type.value,
            "expected_correction": ann.corrected_text,
        })
        # Correct prompt
        rows.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_MSG},
                {"role": "user", "content": TEMPLATES[2].format(
                    source=src, passage=psg,
                    span=ann.hallucinated_span,
                    htype=ann.hallucination_type.value.replace("_", " "))},
            ],
            "task_id": task.task_id.value,
            "expected_action": "correct",
            "expected_span": ann.hallucinated_span,
            "expected_type": ann.hallucination_type.value,
            "expected_correction": ann.corrected_text,
        })
        # Full pipeline prompt
        rows.append({
            "prompt": [
                {"role": "system", "content": SYSTEM_MSG},
                {"role": "user", "content": TEMPLATES[3].format(source=src, passage=psg)},
            ],
            "task_id": task.task_id.value,
            "expected_action": "detect",
            "expected_span": ann.hallucinated_span,
            "expected_type": ann.hallucination_type.value,
            "expected_correction": ann.corrected_text,
        })

# Repeat to get enough training examples (GRPO benefits from more steps)
rows_extended = rows * 8  # 24 base × 8 = 192 prompts

train_dataset = Dataset.from_list(rows_extended)
print(f"✅ Training dataset: {len(train_dataset)} prompts")
print(f"   Tasks: {set(r['task_id'] for r in rows)}")
print(f"   Actions: {set(r['expected_action'] for r in rows)}")
print(f"   Annotations covered: {len(rows) // 4}")

✅ Training dataset: 192 prompts
   Tasks: {'task_hard_multi', 'task_easy_factual', 'task_medium_entity'}
   Actions: {'correct', 'classify', 'detect'}
   Annotations covered: 6


## 🤖 Step 2 — Load Qwen3-0.6B + LoRA

We load `Qwen/Qwen3-0.6B` — the latest-generation Qwen3, fully **ungated** (Apache 2.0).
We use `enable_thinking=False` for structured JSON output (non-thinking mode).
LoRA rank 16 keeps trainable parameters under 2M while being expressive enough for our task.

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3 — Load Model + Tokenizer + LoRA
# ══════════════════════════════════════════════════════════════════════════════

# MODEL_ID is set in Cell 1 (default: "Qwen/Qwen3-0.6B")
print(f"🔄 Loading {MODEL_ID} (ungated — Apache 2.0, no access request needed) ...")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Model — auto dtype selection
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map={"": DEVICE},
    trust_remote_code=True,
)

# LoRA config for GRPO — target all linear layers in Qwen3
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
    bias="none",
)

total = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded: {MODEL_ID}")
print(f"   Total params:     {total:,}")
print(f"   Device: {next(model.parameters()).device}")
print(f"   Dtype: {next(model.parameters()).dtype}")
print(f"   LoRA rank: 16  |  LoRA alpha: 32")

🔄 Loading Qwen/Qwen3-0.6B (ungated — Apache 2.0, no access request needed) ...


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

✅ Model loaded: Qwen/Qwen3-0.6B
   Total params:     596,049,920
   Device: mps:0
   Dtype: torch.bfloat16
   LoRA rank: 16  |  LoRA alpha: 32


## 🎯 Step 3 — Define GRPO Reward Functions

The reward function is the core of GRPO. We define **3 reward signals** that directly use 
the gym's `RewardEngine`:

1. **Format reward** — Is the output valid JSON with correct fields?
2. **Detection reward** — Does it correctly identify the hallucinated span?
3. **Classification + Correction reward** — Correct type + good correction?

GRPO samples multiple completions per prompt and ranks them by reward to compute the policy gradient.

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4 — GRPO Reward Functions
# ══════════════════════════════════════════════════════════════════════════════
from hallucination_detector_gym.rewards import RewardEngine, _span_overlap_ratio
from hallucination_detector_gym.models import HallucinationAction
from hallucination_detector_gym.constants import ActionType

def _extract_json(text: str) -> dict:
    """Try to extract a JSON object from model output."""
    if not text:
        return {}
    # Try direct parse
    try:
        return json.loads(text.strip())
    except Exception:
        pass
    # Try code block
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    # Find any JSON
    m = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return {}


def _completion_to_text(completion) -> str:
    """Extract raw text from a TRL completion.

    In TRL ≥1.0 conversational mode, each completion is a list of message dicts
    like [{"role": "assistant", "content": "..."}].
    In non-conversational mode it is already a plain string.
    This helper normalises both cases to a single string.
    """
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        parts = []
        for msg in completion:
            if isinstance(msg, dict):
                parts.append(str(msg.get("content", "")))
            else:
                parts.append(str(msg))
        return " ".join(parts)
    return str(completion)


def reward_format(completions, **kwargs) -> list[float]:
    """Reward 1: Is the output valid JSON with the required action_type field?"""
    rewards = []
    for comp in completions:
        text = _completion_to_text(comp)
        obj = _extract_json(text)
        if not obj:
            rewards.append(-1.0)
            continue
        score = 0.0
        if "action_type" in obj and obj["action_type"] in ("detect", "classify", "correct", "submit"):
            score += 0.5
        if obj.get("reasoning"):
            score += 0.1
        if obj.get("hallucinated_span"):
            score += 0.2
        if obj.get("hallucination_type") in ("factual_error", "entity_fabrication", "logical_inconsistency", "none"):
            score += 0.1
        if obj.get("corrected_text"):
            score += 0.1
        rewards.append(score)
    return rewards


def reward_detection(completions, expected_span, expected_type, **kwargs) -> list[float]:
    """Reward 2: Does the output correctly detect & identify the hallucination?"""
    rewards = []
    for comp, gt_span, gt_type in zip(completions, expected_span, expected_type):
        text = _completion_to_text(comp)
        obj = _extract_json(text)
        if not obj:
            rewards.append(-1.0)
            continue
        score = 0.0
        if obj.get("hallucination_detected") is True:
            score += 0.3
        pred_span = obj.get("hallucinated_span", "")
        if pred_span and gt_span:
            overlap = _span_overlap_ratio(pred_span, gt_span)
            score += 0.4 * overlap
        pred_type = obj.get("hallucination_type", "")
        if pred_type == gt_type:
            score += 0.3
        elif pred_type and pred_type != "none":
            score += 0.05
        rewards.append(score)
    return rewards


def reward_correction(completions, expected_correction, expected_span, **kwargs) -> list[float]:
    """Reward 3: Does the correction match ground truth?"""
    rewards = []
    for comp, gt_fix, gt_span in zip(completions, expected_correction, expected_span):
        text = _completion_to_text(comp)
        obj = _extract_json(text)
        if not obj:
            rewards.append(-0.5)
            continue
        score = 0.0
        pred_fix = obj.get("corrected_text", "")
        if pred_fix and gt_fix:
            overlap = _span_overlap_ratio(pred_fix, gt_fix)
            score += 0.6 * overlap
        pred_span = obj.get("hallucinated_span", "")
        if pred_span and gt_span:
            span_overlap = _span_overlap_ratio(pred_span, gt_span)
            score += 0.4 * span_overlap
        rewards.append(score)
    return rewards


print("✅ 3 reward functions defined (TRL 1.0 compatible):")
print("   1. reward_format     — valid JSON structure (weight: 1.0)")
print("   2. reward_detection  — hallucination detection accuracy (weight: 2.0)")
print("   3. reward_correction — correction quality (weight: 1.5)")
print(f"   ℹ️  _completion_to_text handles both str and conversational msg-list formats")

✅ 3 reward functions defined (TRL 1.0 compatible):
   1. reward_format     — valid JSON structure (weight: 1.0)
   2. reward_detection  — hallucination detection accuracy (weight: 2.0)
   3. reward_correction — correction quality (weight: 1.5)
   ℹ️  _completion_to_text handles both str and conversational msg-list formats


## 📏 Step 4 — Pre-Training Baseline Evaluation

Before GRPO training, we evaluate the **base Qwen3-0.6B** model on all tasks to establish 
a baseline. This lets us measure the exact improvement from fine-tuning.

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5 — Evaluate model on all tasks (used before & after training)
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_model(model_to_eval, tokenizer_to_eval, label: str = "Model") -> dict:
    """Run the model on all gym tasks and compute scores.

    For each task, we run a multi-step episode: detect → classify → correct → submit.
    Returns dict with per-task scores and trajectories.
    """
    model_to_eval.eval()
    results = {}

    for task in tasks:
        task_id = task.task_id.value
        engine = RewardEngine(task)
        trajectory = []

        for ann in task.annotations:
            for action_name, template_idx in [("detect", 0), ("classify", 1), ("correct", 2)]:
                tmpl = TEMPLATES[template_idx]
                if template_idx == 0:
                    user_msg = tmpl.format(source=task.source_context, passage=task.passage)
                elif template_idx == 1:
                    user_msg = tmpl.format(source=task.source_context, passage=task.passage,
                                           span=ann.hallucinated_span)
                else:
                    user_msg = tmpl.format(source=task.source_context, passage=task.passage,
                                           span=ann.hallucinated_span,
                                           htype=ann.hallucination_type.value.replace("_", " "))

                messages = [
                    {"role": "system", "content": SYSTEM_MSG},
                    {"role": "user", "content": user_msg},
                ]

                input_text = tokenizer_to_eval.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True,
                    enable_thinking=False,
                )
                inputs = tokenizer_to_eval(input_text, return_tensors="pt", truncation=True, max_length=2048)
                inputs = {k: v.to(model_to_eval.device) for k, v in inputs.items()}

                with torch.no_grad():
                    outputs = model_to_eval.generate(
                        **inputs,
                        max_new_tokens=256,
                        temperature=0.1,
                        top_p=0.9,
                        do_sample=True,
                        pad_token_id=tokenizer_to_eval.pad_token_id,
                    )

                gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
                response = tokenizer_to_eval.decode(gen_tokens, skip_special_tokens=True)

                action_dict = _extract_json(response)
                if not action_dict:
                    action_dict = {"action_type": "noop", "reasoning": "Failed to parse"}

                valid_actions = {"detect", "classify", "correct", "submit", "noop"}
                if action_dict.get("action_type") not in valid_actions:
                    action_dict["action_type"] = "noop"

                try:
                    typed_action = HallucinationAction(**action_dict)
                except Exception:
                    typed_action = HallucinationAction(action_type="noop")

                reward, feedback = engine.compute_reward(typed_action)
                trajectory.append({
                    "action": action_name,
                    "reward": reward,
                    "cumulative": engine.cumulative_reward,
                })

        # Submit
        submit_action = HallucinationAction(action_type="submit")
        engine.compute_reward(submit_action)

        final_score = engine.get_final_score()
        results[task_id] = {
            "score": final_score,
            "cumulative_reward": engine.cumulative_reward,
            "trajectory": trajectory,
            "n_steps": len(trajectory),
        }
        print(f"  {label} | {task_id}: score={final_score:.4f}  reward={engine.cumulative_reward:.3f}")

    avg = float(np.mean([r["score"] for r in results.values()]))
    results["average"] = avg
    print(f"  {label} | {'Average':<28s}: {avg:.4f}")
    return results

print(f"\n📏 Evaluating BASE model (before GRPO) ...")
baseline_results = evaluate_model(model, tokenizer, "Baseline")
print(f"\n🏁 Baseline average: {baseline_results['average']:.4f}")


📏 Evaluating BASE model (before GRPO) ...
2026-04-01 01:26:28 [info     ] reward_computed                action_type=detect cumulative=0.3 feedback='Correct detection! Span did not match any known hallucination.' reward=0.3
2026-04-01 01:26:32 [info     ] reward_computed                action_type=classify cumulative=0.2 feedback="Incorrect classification. You said 'entity_fabrication'." reward=-0.1
2026-04-01 01:26:36 [info     ] reward_computed                action_type=correct cumulative=0.2 feedback='Correction did not match expected fix.' reward=0.0
2026-04-01 01:26:36 [info     ] reward_computed                action_type=submit cumulative=0.2 feedback='Episode submitted.' reward=0.0
  Baseline | task_easy_factual: score=0.2000  reward=0.200
2026-04-01 01:26:42 [info     ] reward_computed                action_type=detect cumulative=0.3 feedback='Correct detection! Span did not match any known hallucination.' reward=0.3
2026-04-01 01:26:47 [info     ] reward_computed           

## 🏋️ Step 5 — GRPO Training Loop

Now we fine-tune with GRPO. Key hyperparameters:
- **num_generations = 2** — GRPO samples 2 completions per prompt, ranks by reward (memory-efficient)
- **LoRA rank 16** — lightweight adapters, ~1.5M trainable params
- **beta = 0.04** — KL penalty coefficient (keeps model close to base)
- **3 reward functions** with weights [1.0, 2.0, 1.5] (detection weighted highest)
- **Learning rate = 5e-6** with cosine schedule
- **Auto precision** — uses bf16 on CUDA, fp32 on MPS/CPU

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6 — GRPO Training
# ══════════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = PROJECT_ROOT / "grpo_output"
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Auto-detect precision based on device ─────────────────────────────────────
use_bf16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
use_fp16 = DEVICE == "cuda" and not use_bf16

# ── GRPO Configuration ───────────────────────────────────────────────────────
training_args = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    
    # Training schedule
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    
    # Learning rate
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # GRPO-specific
    num_generations=2,          # Group size: 2 completions per prompt (memory-efficient)
    beta=0.04,                  # KL penalty
    reward_weights=[1.0, 2.0, 1.5],  # format, detection, correction
    
    # Generation config
    max_completion_length=256,
    
    # Precision — auto-detect
    bf16=use_bf16,
    fp16=use_fp16,
    
    # Logging
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    report_to="none",
    
    # Misc
    seed=42,
    remove_unused_columns=False,
    dataloader_num_workers=0,
)

# ── Initialize GRPO Trainer ──────────────────────────────────────────────────
print("🔄 Initializing GRPOTrainer ...")

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=[reward_format, reward_detection, reward_correction],
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(f"✅ GRPOTrainer initialized")
print(f"   Trainable params: {sum(p.numel() for p in trainer.model.parameters() if p.requires_grad):,}")
print(f"   Total params:     {sum(p.numel() for p in trainer.model.parameters()):,}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print(f"   Gradient accum: {training_args.gradient_accumulation_steps}")
print(f"   Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   GRPO generations: {training_args.num_generations}")
print(f"   Reward weights: {training_args.reward_weights}")
print(f"   Precision: bf16={use_bf16}, fp16={use_fp16}")

# ── Train ────────────────────────────────────────────────────────────────────
print("\n🏋️ Starting GRPO training ...")
train_start = time.time()

train_result = trainer.train()

train_elapsed = time.time() - train_start
print(f"\n✅ Training complete in {train_elapsed:.0f}s ({train_elapsed/60:.1f}min)")
print(f"   Final loss: {train_result.training_loss:.4f}")

# Save
trainer.save_model(str(OUTPUT_DIR / "final"))
tokenizer.save_pretrained(str(OUTPUT_DIR / "final"))
print(f"💾 Model saved to {OUTPUT_DIR / 'final'}")

🔄 Initializing GRPOTrainer ...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


✅ GRPOTrainer initialized
   Trainable params: 10,092,544
   Total params:     606,142,464
   Batch size: 1
   Gradient accum: 4
   Effective batch: 4
   Epochs: 3
   GRPO generations: 2
   Reward weights: [1.0, 2.0, 1.5]
   Precision: bf16=False, fp16=False

🏋️ Starting GRPO training ...


Step,Training Loss
5,0.002552
10,0.000714
15,0.000022
20,0.007742
25,0.001840
30,0.000025
35,0.008940
40,0.000028
45,0.000036
50,0.003719



✅ Training complete in 42742s (712.4min)
   Final loss: 0.0044
💾 Model saved to /Users/williyam-23540/Desktop/hallucination-detector-gym/grpo_output/final


## 📊 Step 6 — Post-Training Evaluation & Comparison

Evaluate the GRPO-fine-tuned Qwen3-0.6B model on all tasks and compare against the baseline.

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7 — Post-Training Evaluation
# ══════════════════════════════════════════════════════════════════════════════

print("📏 Evaluating GRPO-fine-tuned model ...")
finetuned_model = trainer.model
finetuned_results = evaluate_model(finetuned_model, tokenizer, "GRPO Fine-tuned")

# ── Comparison Table ─────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("📊 BASELINE vs GRPO FINE-TUNED")
print("=" * 70)
print(f"{'Task':<30s} {'Baseline':>10s} {'GRPO':>10s} {'Δ':>10s}")
print("-" * 62)

task_ids = ["task_easy_factual", "task_medium_entity", "task_hard_multi"]
comparison_data = []

for tid in task_ids:
    b = baseline_results.get(tid, {}).get("score", 0.0)
    g = finetuned_results.get(tid, {}).get("score", 0.0)
    delta = g - b
    print(f"  {tid:<28s} {b:>10.4f} {g:>10.4f} {delta:>+10.4f}")
    comparison_data.append({
        "task_id": tid,
        "baseline": b,
        "grpo": g,
        "delta": delta,
    })

b_avg = baseline_results["average"]
g_avg = finetuned_results["average"]
d_avg = g_avg - b_avg
print("-" * 62)
print(f"  {'AVERAGE':<28s} {b_avg:>10.4f} {g_avg:>10.4f} {d_avg:>+10.4f}")

if b_avg > 0:
    pct = (d_avg / b_avg) * 100
    print(f"\n  📈 Improvement: {pct:+.1f}%")
else:
    print(f"\n  📈 Improvement: {g_avg:.4f} (from zero baseline)")

comparison_data.append({"task_id": "average", "baseline": b_avg, "grpo": g_avg, "delta": d_avg})

📏 Evaluating GRPO-fine-tuned model ...
2026-04-01 13:22:14 [info     ] reward_computed                action_type=detect cumulative=0.3 feedback='Correct detection! Span did not match any known hallucination.' reward=0.3
2026-04-01 13:22:21 [info     ] reward_computed                action_type=detect cumulative=0.8 feedback='Correct detection! Span partially matched (overlap=1.00).' reward=0.5
2026-04-01 13:22:27 [info     ] reward_computed                action_type=correct cumulative=0.8 feedback='Correction did not match expected fix.' reward=0.0
2026-04-01 13:22:27 [info     ] reward_computed                action_type=submit cumulative=0.8 feedback='Episode submitted.' reward=0.0
  GRPO Fine-tuned | task_easy_factual: score=0.8000  reward=0.800
2026-04-01 13:22:37 [info     ] reward_computed                action_type=detect cumulative=0.3 feedback='Correct detection! Span did not match any known hallucination.' reward=0.3
2026-04-01 13:22:46 [info     ] reward_computed          

## 📈 Step 7 — Publication-Quality Training Graphs

Generate comprehensive visualizations showing:
1. **Before vs After** score comparison per task
2. **Training loss curve** from GRPO logs
3. **Reward trajectory** heatmap
4. **Improvement summary** with deltas

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8 — Publication-Quality Visualizations
# ══════════════════════════════════════════════════════════════════════════════

SAVE_DIR = PROJECT_ROOT / "assets"
SAVE_DIR.mkdir(exist_ok=True)

COLORS = {"Easy": "#2ecc71", "Medium": "#f39c12", "Hard": "#e74c3c", "Avg": "#3498db"}
task_labels = {"task_easy_factual": "Easy:\nFactual Error",
               "task_medium_entity": "Medium:\nEntity+Factual",
               "task_hard_multi": "Hard:\nMulti-type"}

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Before vs After Score Comparison (main result)
# ══════════════════════════════════════════════════════════════════════════════
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={"width_ratios": [2, 1]})
fig.patch.set_facecolor("white")
fig.suptitle("🏆 GRPO Fine-Tuning Results — Hallucination Detection (Qwen3-0.6B)", 
             fontsize=17, fontweight="bold", y=1.01)

# Left: Grouped bar chart
x = np.arange(len(task_ids))
width = 0.32

baselines = [baseline_results.get(t, {}).get("score", 0) for t in task_ids]
grpo_scores = [finetuned_results.get(t, {}).get("score", 0) for t in task_ids]

bars1 = ax1.bar(x - width/2, baselines, width, label="Base Qwen3-0.6B",
                color="#bdc3c7", edgecolor="white", linewidth=1.5, zorder=3)
bars2 = ax1.bar(x + width/2, grpo_scores, width, label="GRPO Fine-tuned",
                color="#3498db", edgecolor="white", linewidth=1.5, zorder=3)

# Value labels
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., h + 0.02,
                 f"{h:.3f}", ha="center", va="bottom", fontweight="bold", fontsize=11)

# Delta arrows
for i in range(len(task_ids)):
    delta = grpo_scores[i] - baselines[i]
    color = "#27ae60" if delta > 0 else "#e74c3c"
    ax1.annotate(f"{delta:+.3f}", xy=(x[i] + width/2, grpo_scores[i] + 0.06),
                fontsize=10, fontweight="bold", color=color, ha="center")

ax1.set_xlabel("Task", fontsize=13, fontweight="bold")
ax1.set_ylabel("Score (0–1)", fontsize=13, fontweight="bold")
ax1.set_title("Score by Task — Before vs After GRPO", fontsize=14, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels([task_labels[t] for t in task_ids], fontsize=10)
ax1.set_ylim(0, 1.15)
ax1.legend(loc="upper right", fontsize=11, framealpha=0.9)
ax1.grid(True, alpha=0.3, axis="y")
ax1.axhline(y=1.0, color="green", linestyle=":", alpha=0.3)

# Right: Average comparison
avg_data = [b_avg, g_avg]
avg_colors = ["#bdc3c7", "#3498db"]
avg_labels = ["Baseline", "GRPO"]
bars_avg = ax2.bar(range(2), avg_data, 0.6, color=avg_colors, edgecolor="white", linewidth=2, zorder=3)

for bar, val in zip(bars_avg, avg_data):
    ax2.text(bar.get_x() + bar.get_width()/2., val + 0.02,
             f"{val:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=14)

ax2.set_ylabel("Average Score", fontsize=13, fontweight="bold")
ax2.set_title("Average Score", fontsize=14, fontweight="bold")
ax2.set_xticks(range(2))
ax2.set_xticklabels(avg_labels, fontsize=12, fontweight="bold")
ax2.set_ylim(0, 1.15)
ax2.grid(True, alpha=0.3, axis="y")

# Improvement arrow
improvement_text = f"{d_avg:+.4f}"
if b_avg > 0:
    improvement_text += f"\n({(d_avg/b_avg)*100:+.1f}%)"
ax2.annotate(improvement_text, xy=(0.5, max(b_avg, g_avg) + 0.08),
            fontsize=13, fontweight="bold", color="#27ae60" if d_avg > 0 else "#e74c3c",
            ha="center")

plt.tight_layout(pad=3)
fig.savefig(SAVE_DIR / "training_results.png", dpi=200, bbox_inches="tight", facecolor="white")
print(f"💾 Saved: {SAVE_DIR / 'training_results.png'}")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Training Loss Curve (from trainer logs)
# ══════════════════════════════════════════════════════════════════════════════
fig2, ax_loss = plt.subplots(figsize=(12, 5))
fig2.patch.set_facecolor("white")

log_history = trainer.state.log_history
train_losses = [(l["step"], l["loss"]) for l in log_history if "loss" in l]

if train_losses:
    steps_l, losses = zip(*train_losses)
    ax_loss.plot(steps_l, losses, "o-", color="#e74c3c", linewidth=2, markersize=5, label="Training Loss")
    
    # Smoothed
    if len(losses) > 3:
        window = min(5, len(losses))
        smoothed = pd.Series(losses).rolling(window=window, min_periods=1).mean()
        ax_loss.plot(steps_l, smoothed, "-", color="#2c3e50", linewidth=3, alpha=0.7, label=f"Smoothed (window={window})")
    
    ax_loss.set_xlabel("Training Step", fontsize=13, fontweight="bold")
    ax_loss.set_ylabel("Loss", fontsize=13, fontweight="bold")
    ax_loss.set_title("📉 GRPO Training Loss Curve (Qwen3-0.6B)", fontsize=15, fontweight="bold")
    ax_loss.legend(fontsize=11)
    ax_loss.grid(True, alpha=0.3)
else:
    ax_loss.text(0.5, 0.5, "No training loss data available", ha="center", va="center", fontsize=14)
    ax_loss.set_title("📉 GRPO Training Loss Curve", fontsize=15, fontweight="bold")

plt.tight_layout()
fig2.savefig(SAVE_DIR / "training_loss.png", dpi=200, bbox_inches="tight", facecolor="white")
print(f"💾 Saved: {SAVE_DIR / 'training_loss.png'}")
plt.show()

# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Reward Comparison Heatmap
# ══════════════════════════════════════════════════════════════════════════════
fig3, ax3 = plt.subplots(figsize=(10, 4))
fig3.patch.set_facecolor("white")

heatmap_data = {
    "Baseline": [baseline_results.get(t, {}).get("score", 0) for t in task_ids],
    "GRPO Fine-tuned": [finetuned_results.get(t, {}).get("score", 0) for t in task_ids],
}
df_heat = pd.DataFrame(heatmap_data, index=["Easy", "Medium", "Hard"]).T

sns.heatmap(df_heat, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0, vmax=1,
            linewidths=2, linecolor="white", cbar_kws={"label": "Score"},
            annot_kws={"fontsize": 16, "fontweight": "bold"}, ax=ax3)

ax3.set_title("🎯 Score Heatmap: Baseline vs GRPO (Qwen3-0.6B)", fontsize=15, fontweight="bold", pad=15)
ax3.set_ylabel("")
ax3.set_xlabel("Task Difficulty", fontsize=12)

plt.tight_layout()
fig3.savefig(SAVE_DIR / "score_heatmap.png", dpi=200, bbox_inches="tight", facecolor="white")
print(f"💾 Saved: {SAVE_DIR / 'score_heatmap.png'}")
plt.show()

print(f"\n✅ All figures saved to {SAVE_DIR}/")

💾 Saved: /Users/williyam-23540/Desktop/hallucination-detector-gym/assets/training_results.png
💾 Saved: /Users/williyam-23540/Desktop/hallucination-detector-gym/assets/training_loss.png
💾 Saved: /Users/williyam-23540/Desktop/hallucination-detector-gym/assets/score_heatmap.png

✅ All figures saved to /Users/williyam-23540/Desktop/hallucination-detector-gym/assets/


## 🚀 Step 8 — Save Results & Push to Hugging Face

Save training artifacts and push the GRPO-fine-tuned LoRA adapter + results to Hugging Face Hub.

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9 — Save Results & Push to Hugging Face
# ══════════════════════════════════════════════════════════════════════════════
import requests

# ── Save training results JSON locally ────────────────────────────────────────
training_output = {
    "model": MODEL_ID,
    "method": "GRPO (Group Relative Policy Optimization)",
    "training_date": datetime.now().isoformat(),
    "training_duration_seconds": round(train_elapsed, 2),
    "device": DEVICE,
    "lora_rank": 16,
    "lora_alpha": 32,
    "num_generations": 2,
    "beta": 0.04,
    "epochs": 3,
    "learning_rate": 5e-6,
    "baseline_results": {
        tid: {"score": baseline_results.get(tid, {}).get("score", 0)}
        for tid in task_ids
    },
    "baseline_average": round(float(b_avg), 4),
    "grpo_results": {
        tid: {"score": finetuned_results.get(tid, {}).get("score", 0)}
        for tid in task_ids
    },
    "grpo_average": round(float(g_avg), 4),
    "improvement": round(float(d_avg), 4),
    "improvement_pct": round(float((d_avg / b_avg) * 100), 1) if b_avg > 0 else None,
}

results_path = PROJECT_ROOT / "training_results.json"
with open(results_path, "w") as f:
    json.dump(training_output, f, indent=2)
print(f"💾 Saved: {results_path}")

# ── Push to Hugging Face ─────────────────────────────────────────────────────
from huggingface_hub import HfApi, create_repo, whoami

HF_TOKEN_W = os.getenv("HF_TOKEN", "")

if not HF_TOKEN_W:
    print("⚠️  HF_TOKEN not set — skipping HF push.")
    print("   To deploy: add HF_TOKEN (write) and HF_USERNAME to your .env file.")
elif not MODEL_REPO:
    print("⚠️  HF_USERNAME not set — skipping HF push.")
    print("   To deploy: add HF_USERNAME to your .env file.")
else:
    # 1. Validate token
    print("🔑 Validating HF token ...")
    try:
        user_info = whoami(token=HF_TOKEN_W)
        print(f"   ✅ Authenticated as: {user_info['name']}")
    except Exception as e:
        raise RuntimeError(f"HF token invalid: {e}. Fix HF_TOKEN in .env")

    api = HfApi(token=HF_TOKEN_W)

    # 2. Create/ensure repo exists
    print(f"\n📦 Creating model repo: {MODEL_REPO} ...")
    create_repo(repo_id=MODEL_REPO, token=HF_TOKEN_W, repo_type="model",
                exist_ok=True, private=False)
    print(f"   ✅ Repo ready: https://huggingface.co/{MODEL_REPO}")

    # 3. Build model card
    b_scores = training_output["baseline_results"]
    g_scores = training_output["grpo_results"]

    model_card = f"""---
tags:
  - hallucination-detection
  - grpo
  - qwen3
  - lora
  - trl
  - openenv
  - ai-safety
license: apache-2.0
base_model: {MODEL_ID}
language:
  - en
metrics:
  - accuracy
library_name: peft
model-index:
  - name: Hallucination Detector GRPO (Qwen3-0.6B)
    results:
      - task:
          type: hallucination-detection
          name: Hallucination Detection
        metrics:
          - name: Average Score (GRPO)
            type: accuracy
            value: {g_avg:.4f}
          - name: Average Score (Baseline)
            type: accuracy
            value: {b_avg:.4f}
---

# 🧠 Hallucination Detector Agent — GRPO Fine-tuned Qwen3-0.6B

A **GRPO-fine-tuned** LoRA adapter for hallucination detection, classification, and correction.

## Quick Start

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model + LoRA adapter
base = AutoModelForCausalLM.from_pretrained("{MODEL_ID}", torch_dtype="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, "{MODEL_REPO}")
tokenizer = AutoTokenizer.from_pretrained("{MODEL_ID}", trust_remote_code=True)

messages = [
    {{"role": "system", "content": "You are a hallucination detector. Given SOURCE and PASSAGE, output JSON with action_type, hallucination_detected, hallucination_type, hallucinated_span, corrected_text, reasoning."}},
    {{"role": "user", "content": "SOURCE: The Eiffel Tower is in Paris.\\nPASSAGE: The Eiffel Tower is in London."}}
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```

## Model Details

| | |
|---|---|
| **Base model** | [`{MODEL_ID}`](https://huggingface.co/{MODEL_ID}) |
| **License** | Apache 2.0 (ungated) |
| **Method** | GRPO (Group Relative Policy Optimization) |
| **LoRA** | rank=16, alpha=32 |
| **Training** | 3 epochs, lr=5e-6, beta=0.04, 2 generations/prompt |

## Results

| Task | Baseline | GRPO | Δ |
|---|---|---|---|
| Easy: Factual Error | {b_scores['task_easy_factual']['score']:.4f} | {g_scores['task_easy_factual']['score']:.4f} | {g_scores['task_easy_factual']['score'] - b_scores['task_easy_factual']['score']:+.4f} |
| Medium: Entity+Factual | {b_scores['task_medium_entity']['score']:.4f} | {g_scores['task_medium_entity']['score']:.4f} | {g_scores['task_medium_entity']['score'] - b_scores['task_medium_entity']['score']:+.4f} |
| Hard: Multi-type | {b_scores['task_hard_multi']['score']:.4f} | {g_scores['task_hard_multi']['score']:.4f} | {g_scores['task_hard_multi']['score'] - b_scores['task_hard_multi']['score']:+.4f} |
| **Average** | **{b_avg:.4f}** | **{g_avg:.4f}** | **{d_avg:+.4f}** |

### Training Plots
![Training Results](training_results.png)
![Training Loss](training_loss.png)
![Score Heatmap](score_heatmap.png)

## Reproduce

```bash
git clone https://github.com/your-username/hallucination-detector-gym
cd hallucination-detector-gym
cp .env.example .env  # Add your HF_TOKEN and HF_USERNAME
pip install -e .
# Open training_hallucination_detector.ipynb and run all cells
```

## Framework
- **TRL** + **PEFT** + **Transformers** (latest)
- Trained on {DEVICE} in {train_elapsed:.0f}s
"""

    readme_path = PROJECT_ROOT / "model_card.md"
    with open(readme_path, "w") as f:
        f.write(model_card)

    # 4. Upload LoRA adapter
    print(f"\n📤 Uploading LoRA adapter to {MODEL_REPO} ...")
    model_dir = OUTPUT_DIR / "final"
    if model_dir.exists():
        api.upload_folder(
            folder_path=str(model_dir),
            repo_id=MODEL_REPO,
            repo_type="model",
            token=HF_TOKEN_W,
            commit_message="Upload GRPO-trained LoRA adapter (Qwen3-0.6B)",
        )
        print("   ✅ LoRA adapter uploaded")
    else:
        print(f"   ❌ No adapter at {model_dir} — training may not have saved correctly")

    # 5. Upload model card + plots + results
    print("\n📤 Uploading model card & plots ...")
    for fname, local_path in [
        ("README.md", str(readme_path)),
        ("training_results.json", str(results_path)),
        ("training_results.png", str(SAVE_DIR / "training_results.png")),
        ("training_loss.png", str(SAVE_DIR / "training_loss.png")),
        ("score_heatmap.png", str(SAVE_DIR / "score_heatmap.png")),
    ]:
        if Path(local_path).exists():
            try:
                api.upload_file(path_or_fileobj=local_path, path_in_repo=fname,
                               repo_id=MODEL_REPO, repo_type="model", token=HF_TOKEN_W)
                print(f"   ✅ {fname}")
            except Exception as e:
                print(f"   ⚠️ {fname}: {e}")

    # 6. Validate — confirm repo is accessible (not 404)
    print(f"\n🔍 Validating deployment ...")
    api_url = f"https://huggingface.co/api/models/{MODEL_REPO}"
    try:
        resp = requests.get(api_url, timeout=15)
        if resp.status_code == 200:
            data = resp.json()
            files = [s.get("rfilename", "") for s in data.get("siblings", [])]
            has_adapter = any("adapter" in f for f in files)
            has_readme = "README.md" in files
            print(f"   ✅ Repo live: https://huggingface.co/{MODEL_REPO}")
            print(f"   {'✅' if has_adapter else '❌'} adapter_model found")
            print(f"   {'✅' if has_readme else '❌'} README.md found")
            print(f"   📁 Files: {files}")
        else:
            print(f"   ❌ HTTP {resp.status_code} — repo not accessible!")
    except Exception as e:
        print(f"   ⚠️ Validation failed: {e}")

    # Cleanup temp model card
    if readme_path.exists():
        readme_path.unlink()

    print(f"\n🚀 Model published: https://huggingface.co/{MODEL_REPO}")

💾 Saved: /Users/williyam-23540/Desktop/hallucination-detector-gym/training_results.json
🔑 Validating HF token ...
   ✅ Authenticated as: williyam

📦 Creating model repo: williyam/hallucination-detector-agent-qwen3-0.6b ...
   ✅ Repo ready: https://huggingface.co/williyam/hallucination-detector-agent-qwen3-0.6b

📤 Uploading LoRA adapter to williyam/hallucination-detector-agent-qwen3-0.6b ...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

   ✅ LoRA adapter uploaded

📤 Uploading model card & plots ...
   ✅ README.md
   ✅ training_results.json


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

   ✅ training_results.png


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

   ✅ training_loss.png
   ✅ score_heatmap.png

🔍 Validating deployment ...
   ✅ Repo live: https://huggingface.co/williyam/hallucination-detector-agent-qwen3-0.6b
   ✅ adapter_model found
   ✅ README.md found
   📁 Files: ['.gitattributes', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'score_heatmap.png', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'training_loss.png', 'training_results.json', 'training_results.png']

🚀 Model published: https://huggingface.co/williyam/hallucination-detector-agent-qwen3-0.6b


## ✅ Summary

### What We Built
A **complete GRPO fine-tuning pipeline** that:
1. Uses **Qwen3-0.6B** — latest-generation Qwen, ungated (Apache 2.0)
2. Applies **GRPO** (Group Relative Policy Optimization) from TRL — no critic model needed
3. Trains with **3 reward functions** from the gym's RewardEngine
4. Uses **LoRA rank 16** — only ~1.5M trainable parameters
5. Uses `enable_thinking=False` for structured JSON output (non-thinking mode)
6. Evaluates **before and after** on all 3 difficulty levels
7. Generates **publication-quality plots**
8. Pushes **LoRA adapter** to Hugging Face Hub as `hallucination-detector-agent-qwen3-0.6b`

### Why GRPO > Other Methods
| Method | Needs Critic? | Memory | Quality |
|--------|:---:|:---:|:---:|
| PPO | ✅ Yes | 2× model | Good |
| DPO | ❌ No | 1× model | Good (needs pairs) |
| **GRPO** | ❌ No | 1× model | **Best** (multi-sample ranking) |
| REINFORCE | ❌ No | 1× model | High variance |

GRPO achieves the best of both worlds: no critic overhead like PPO, but lower variance 
than REINFORCE by using group-relative baselines.